In [ ]:
import scripts.helpers as helpers

1. Step 1 - Create unified labels file -> in parquete format
2. Step 2 - Unify each SWC tree with it's labels
3. Step 3 - Simplify SWC tree
4. Step 4 - Calculate clumpiness for each node in the tree, for each labels combination
5. Step 5 - Insert the clumpiness score  into the simplified SWC file

---
# Step 1 - Create unified labels file -> in parquete format

---
# Step 2 - Unify each SWC tree with it's labels

---
# Step 3 - Simplify SWC tree

---
# Step 4 - Calculate clumpiness for each node in the tree, for each labels combination

---
# Step 5 - Insert the clumpiness score  into the simplified SWC file

---
# Preprocessing step
1. Create metadata labels for each swc file
2. Look for the releveant swc files only (with the wanted type)
3. Unify them via the already created function in feather file ->>> Improvement

In [2]:
import os
import pandas as pd
from tqdm import tqdm
from scripts.helpers import mkdir


# Type data located in the 

path_swc_labels = os.path.join("data", "input_labels", "neuron_data_full_article_princeton.ftr")
swc_labels = pd.read_feather(path_swc_labels)

required_labels = ["super_class", ["central", "optic", "visual_centrifugal", "visual_projection"]]
swc_labels = swc_labels.loc[swc_labels[required_labels[0]].isin(required_labels[1])]

In [6]:
def get_neurons_info(main_path : str = os.path.join("data", "input_labels"),
                     swc_labels_filter : str = (True, ("super_class",["central", "optic", "visual_centrifugal", "visual_projection"])),
                     swc_labels_file : str = "neuron_data_full_article_princeton.ftr",
                     nodes_labels_folder : str = "processed_swc_data_princeton",
                     nodes_labels_name : str = "connectors.pkl",
                     overwrite_parquet : bool = True
                    ) -> pd.DataFrame:
    """
    main_path : str -> path to the folder which contains all the labels required files.
    swc_labels_filter : str -> if [0] is True filter out unwanted swc file on the base of their super_class label [1][0] and their super type labels rquired [1][1] of the touple.
    swc_labels_file : str -> name of the neurons swc labels file.
    nodes_labels_folder : str -> name of the folders containing the nodes labels data.
    nodes_labels_name : str -> name of the file which contains the nodes labels (pre / post synaptic).
    overwrite_parquet : bool ->
    """

    parquet_path = os.path.join(main_path, "swc_labels.parquet")
    if (overwrite_parquet is False) & os.path.exists(parquet_path):
        raise Exception("> Function execution halted, old `swc_labels.parquet` file preserved.")

    else:
        os.remove(parquet_path)
        print("> Old `swc_labels.parquet` deleted, creating a new file.")
        
    # 1. Generating a required neurons dataframe with their super-class type.
    if swc_labels_filter:
        try:
            filter_on = swc_labels_filter[1][0]
            filter_by = swc_labels_filter[1][1]

            swc_labels = pd.read_feather(os.path.join(main_path, swc_labels_file))
            swc_labels = swc_labels.loc[swc_labels[filter_on].isin(filter_by), ["neuron", filter_on]].drop_duplicates()
            swc_labels.neuron = swc_labels.neuron.astype("str")

        except:
            raise Exception("> Error occured while tying to generate relevent neurons labels, please cheeck `swc_labels_filter` argument input.")


    # 2. Loading the nodes labels files, cheecking for required neurons and saving the data to parquet (concat) with each itiration for storage efficincy.
    # Mapping the folders
    
    nodes_labels_path = os.path.join(main_path, nodes_labels_folder)
    folders = os.listdir(nodes_labels_path)
    paths = []


    # Getting list of folders with the connector file
    for i in tqdm(folders, desc="Procssing metadata files", unit="files"):
        i_path = os.path.join(nodes_labels_path, i, nodes_labels_name)
        if os.path.exists(i_path):
            temp_labels = pd.read_pickle(i_path)
            temp_labels.neuron = temp_labels.neuron.astype("str")

            try:
                temp_labels = pd.merge(left=temp_labels, 
                                       right=swc_labels, 
                                       left_on="neuron",
                                       right_on="neuron",
                                       how="inner")

            except:
                pass

        
        
        if os.path.exists(parquet_path) is False:
            temp_labels.to_parquet(parquet_path, 
                                   engine="fastparquet", 
                                   compression="zstd")
        else:
            temp_labels.to_parquet(parquet_path, 
                                   engine="fastparquet", 
                                   compression="zstd",
                                   append=True)


labels_swc = get_neurons_info()


> Old `swc_labels.parquet` deleted, creating a new file.


Procssing metadata files: 100%|██████████| 93/93 [04:45<00:00,  3.07s/files]
